In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:
if torch.cuda.is_available():
    device=torch.device("cuda")
    print("using gpu")
else:
    device=torch.device("cpu")
    print("using cpu")

using gpu


In [ ]:
# The train_dataset does not contain already processed files. The files on your disk, which are loaded by ImageFolder, are in their original format.

# The train_loader isn't performing the transformations directly. Instead, the train_dataset (which is an ImageFolder instance) applies the transform you defined in cell BX-T7KaECWCe every time an image is requested by the DataLoader.

# So, when the DataLoader iterates and fetches a batch:

# It asks the train_dataset for images.
# For each image, the train_dataset reads the raw image file from the specified path (/content/archive/seg_train/seg_train).
# It then applies the transform sequence to that raw image:
# transforms.Resize((150,150)): Resizes the image to 150x150 pixels.
# transforms.ToTensor(): Converts the PIL Image or numpy array to a PyTorch Tensor and scales the pixel values to the range [0.0, 1.0].
# These transformed images are then batched together and yielded by the DataLoader.

In [ ]:
transform=transforms.Compose([
    transforms.Resize((150,150)),
    transforms.ToTensor(), # Convert to 3 channels and normalize to 0-1
])

# ImageFolder will load all folders as different classes
train_dataset=ImageFolder(
    root=r'archive\seg_train\seg_train',#For local Training
    transform=transform
)

train_loader=DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4 #Could show warning as the dataset is taking 4 worker to fetch data from dataloader parallel
    # (if training stop use 0,1,2 worker)
)

print(train_dataset.classes)

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
for image,labels in train_loader:
    print(image.size())
    print(image)
    print(labels)
    break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


torch.Size([16, 3, 150, 150])
tensor([[[[0.6078, 0.6039, 0.6118,  ..., 0.7490, 0.7490, 0.7451],
          [0.6118, 0.6078, 0.6157,  ..., 0.7294, 0.7255, 0.7216],
          [0.6235, 0.6196, 0.6196,  ..., 0.7216, 0.7137, 0.7137],
          ...,
          [0.1412, 0.1098, 0.1294,  ..., 0.2235, 0.2314, 0.2000],
          [0.1294, 0.1490, 0.1529,  ..., 0.2118, 0.2353, 0.2196],
          [0.1059, 0.2000, 0.1294,  ..., 0.1765, 0.2196, 0.2353]],

         [[0.7176, 0.7137, 0.7098,  ..., 0.8431, 0.8431, 0.8392],
          [0.7216, 0.7176, 0.7137,  ..., 0.8235, 0.8196, 0.8157],
          [0.7216, 0.7176, 0.7216,  ..., 0.8157, 0.8078, 0.8078],
          ...,
          [0.1451, 0.1137, 0.1255,  ..., 0.2863, 0.3020, 0.2824],
          [0.1333, 0.1529, 0.1490,  ..., 0.2627, 0.3020, 0.2902],
          [0.1098, 0.2039, 0.1255,  ..., 0.2314, 0.2784, 0.3020]],

         [[0.8706, 0.8667, 0.8667,  ..., 0.9373, 0.9373, 0.9333],
          [0.8745, 0.8706, 0.8706,  ..., 0.9176, 0.9137, 0.9098],
          [0

In [ ]:
class CNN(nn.Module):

    def __init__(self, input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
epoches=125
learning_rate=0.001

In [ ]:
model = CNN(3)
model = model.to(device)
# loss
criteria = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
#Training loop
for epoch in range(epoches):
    total_epoch_loss=0
    for batch_features,batch_label in train_loader:

        #moving data to gpu
        batch_features,batch_label=batch_features.to(device),batch_label.to(device)

        #This is forward pass
        z=model.forward(batch_features)

        #This is loss calculation
        loss=criteria(z,batch_label)

        #backward pass
        optimizer.zero_grad()
        loss.backward()

        #update grads
        optimizer.step()

        total_epoch_loss=total_epoch_loss+loss.item()

    average_loss=total_epoch_loss/len(train_loader)
    print(f"Epoch: {epoch+1}  Loss: {average_loss}")

Epoch: 1  Loss: 0.9482895252984315
Epoch: 2  Loss: 0.739726542764617
Epoch: 3  Loss: 0.6476977000136038
Epoch: 4  Loss: 0.5953430628280977
Epoch: 5  Loss: 0.5574010616520543
Epoch: 6  Loss: 0.5325099293117099
Epoch: 7  Loss: 0.5031284658251021
Epoch: 8  Loss: 0.48522418013568075
Epoch: 9  Loss: 0.45952328495630373
Epoch: 10  Loss: 0.4380165870536908
Epoch: 11  Loss: 0.41875998293959743
Epoch: 12  Loss: 0.40173890082280156
Epoch: 13  Loss: 0.3880101794118151
Epoch: 14  Loss: 0.3742236754451804
Epoch: 15  Loss: 0.3677553724421184
Epoch: 16  Loss: 0.3478803869913882
Epoch: 17  Loss: 0.34613794718862667
Epoch: 18  Loss: 0.32025078847444277
Epoch: 19  Loss: 0.31603577101692254
Epoch: 20  Loss: 0.30919222471331814
Epoch: 21  Loss: 0.29512161593078073
Epoch: 22  Loss: 0.286350173379267
Epoch: 23  Loss: 0.27370988280387226
Epoch: 24  Loss: 0.26532268147397964
Epoch: 25  Loss: 0.25916117281602075
Epoch: 26  Loss: 0.244102875445386
Epoch: 27  Loss: 0.2369722184218076
Epoch: 28  Loss: 0.228120763

In [ ]:
torch.save(model.state_dict(),"Img_Classifier.pth")